### Final Project: Detection of Rare Fetal Congenital Heart Diseases (CHD) Using CARDIUM Ultrasound Images and FetalCLIP: A Deep-Learning Model for Rural Coloradans
#### CSCI 5922
#### Spring 2026
#### Meghna Nag and Vanessa Thorsten

# CSCI5922_CARDIUM_Multimodal_Embeddings.ipynb
# Goal:
# 1. Read saved image embeddings and saved tabular embeddings
# 2. Align image-level samples with patient-level tabular embeddings
# 3. Train a multimodal fusion model directly on embeddings
# 4. Compare BCE, Weighted BCE, and Focal Loss
# 5. Evaluate at the patient level using mean pooling

#1. Imports

In [1]:
import os
import re
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# 2. Config

In [3]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

@dataclass
class Config:
    # Root paths
    cardium_root:               str  = "/content/gdrive/MyDrive/CARDIUM"
    image_embeddings_dirname:   str  = "CARDIUM_image_embeddings"
    tabular_embeddings_dirname: str  = "CARDIUM_tabular_embeddings"
    output_root:                str  = "/content/gdrive/MyDrive/CARDIUM/outputs_multimodal"

    # Fold-specific file naming
    image_embedding_pattern:   str = "cardium_fetalclip_embeddings_fold_{fold_idx}.parquet"
    tabular_embedding_pattern: str = "cardium_tabular_embeddings_fold{tab_fold_idx}.parquet"

    # Data columns
    image_id_col:    str = "image_id"
    patient_id_col:  str = "patient_id"
    label_col:       str = "label"
    split_col:       str = "split"
    image_index_col: str = "image_index"
    gest_week_col:   str = "gestational_week_of_imaging"

    # Fold / validation setup
    n_folds:      int   = 3
    val_size:     float = 0.15
    random_state: int   = 42

    # Patient-level image selection
    # FIXED: None means use ALL images per patient — no artificial cap
    max_images_per_patient: Optional[int] = None
    selection_strategy:     str           = "all"

    # Model
    fusion_model_name:      str           = "cross_attention_patient"
    img_dim:                Optional[int] = None
    tab_dim:                Optional[int] = None
    hidden_dim:             int           = 256
    embed_dim:              int           = 256
    num_heads:              int           = 4
    num_layers:             int           = 2
    dropout:                float         = 0.2
    classifier_hidden_dim:  int           = 128

    # Loss
    loss_name:    str            = "weighted_bce"
    pos_weight:   Optional[float] = None
    focal_alpha:  float           = 0.25
    focal_gamma:  float           = 2.0
    use_patient_weighting: bool   = False

    # Training
    batch_size:         int   = 16      # reduced from 32 because sequences are now longer
    lr:                 float = 1e-4    # FIXED: was 1e-3, too aggressive for frozen embeddings
    weight_decay:       float = 1e-4
    max_epochs:         int   = 30
    patience:           int   = 6
    decision_threshold: float = 0.5
    # FIXED: was implicitly val_recall — easily gamed by always predicting positive.
    # pr_auc cannot be gamed: it measures the full precision-recall tradeoff
    best_metric: str = "pr_auc"

    # Runtime
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


CFG = Config()
Path(CFG.output_root).mkdir(parents=True, exist_ok=True)
print(CFG)

Config(cardium_root='/content/gdrive/MyDrive/CARDIUM', image_embeddings_dirname='CARDIUM_image_embeddings', tabular_embeddings_dirname='CARDIUM_tabular_embeddings', output_root='/content/gdrive/MyDrive/CARDIUM/outputs_multimodal', image_embedding_pattern='cardium_fetalclip_embeddings_fold_{fold_idx}.parquet', tabular_embedding_pattern='cardium_tabular_embeddings_fold{tab_fold_idx}.parquet', image_id_col='image_id', patient_id_col='patient_id', label_col='label', split_col='split', image_index_col='image_index', gest_week_col='gestational_week_of_imaging', n_folds=3, val_size=0.15, random_state=42, max_images_per_patient=None, selection_strategy='all', fusion_model_name='cross_attention_patient', img_dim=None, tab_dim=None, hidden_dim=256, embed_dim=256, num_heads=4, num_layers=2, dropout=0.2, classifier_hidden_dim=128, loss_name='weighted_bce', pos_weight=None, focal_alpha=0.25, focal_gamma=2.0, use_patient_weighting=False, batch_size=16, lr=0.0001, weight_decay=0.0001, max_epochs=30, 

Reproducibility

In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.random_state)

Helper functions

In [5]:
def parse_image_id(image_id: str) -> Tuple[Optional[int], str]:
    """
    Example:
      0_12345 -> (0, '12345')
      1_12345 -> (1, '12345')
    """
    image_id = str(image_id)
    match = re.match(r"^(\d+)_+(.+)$", image_id)
    if match:
        return int(match.group(1)), match.group(2)
    return None, image_id


def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

    try:
        metrics["auc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["auc"] = np.nan

    try:
        metrics["pr_auc"] = average_precision_score(y_true, y_prob)
    except ValueError:
        metrics["pr_auc"] = np.nan

    return metrics

PathManager

In [6]:
class PathManager:
    def __init__(self, config: Config):
        self.cfg = config

    def get_image_embeddings_root(self) -> Path:
        return Path(self.cfg.cardium_root) / self.cfg.image_embeddings_dirname

    def get_tabular_embeddings_root(self) -> Path:
        return Path(self.cfg.cardium_root) / self.cfg.tabular_embeddings_dirname

    def get_image_embedding_path(self, fold_idx: int) -> Path:
        file_name = self.cfg.image_embedding_pattern.format(fold_idx=fold_idx)
        return self.get_image_embeddings_root() / file_name

    def get_tabular_embedding_path(self, fold_idx: int) -> Path:
        # image folds are 1,2,3 but your tab files appear 0,1,2
        tab_fold_idx = fold_idx - 1
        file_name = self.cfg.tabular_embedding_pattern.format(tab_fold_idx=tab_fold_idx)
        return self.get_tabular_embeddings_root() / file_name

    def validate_paths(self, fold_idx: int):
        img_path = self.get_image_embedding_path(fold_idx)
        tab_path = self.get_tabular_embedding_path(fold_idx)

        print("Image path:", img_path)
        print("Exists?", img_path.exists())
        print("Tabular path:", tab_path)
        print("Exists?", tab_path.exists())

        if not img_path.exists():
            raise FileNotFoundError(f"Missing image embedding file: {img_path}")
        if not tab_path.exists():
            raise FileNotFoundError(f"Missing tabular embedding file: {tab_path}")

EmbeddingLoader

In [7]:
class EmbeddingLoader:
    def __init__(self, config: Config, path_manager: PathManager):
        self.cfg = config
        self.path_manager = path_manager

    def load_image_embeddings(self, fold_idx: int) -> pd.DataFrame:
        path = self.path_manager.get_image_embedding_path(fold_idx)
        df = pd.read_parquet(path)

        print("Loaded image embeddings:", df.shape)
        print("Image columns:", df.columns.tolist())

        if self.cfg.image_id_col not in df.columns:
            raise ValueError(
                f"Expected column '{self.cfg.image_id_col}' in image embeddings, "
                f"but got: {df.columns.tolist()}"
            )

        if self.cfg.patient_id_col not in df.columns:
            parsed = df[self.cfg.image_id_col].apply(parse_image_id)
            df[self.cfg.image_index_col] = [x[0] for x in parsed]
            df[self.cfg.patient_id_col] = [x[1] for x in parsed]
        else:
            if self.cfg.image_index_col not in df.columns:
                parsed = df[self.cfg.image_id_col].apply(parse_image_id)
                df[self.cfg.image_index_col] = [x[0] for x in parsed]

        return df

    def load_tabular_embeddings(self, fold_idx: int) -> pd.DataFrame:
        path = self.path_manager.get_tabular_embedding_path(fold_idx)
        df = pd.read_parquet(path)

        print("Loaded tabular embeddings:", df.shape)
        print("Tabular columns:", df.columns.tolist())

        if self.cfg.patient_id_col not in df.columns:
            raise ValueError(
                f"Expected column '{self.cfg.patient_id_col}' in tabular embeddings, "
                f"but got: {df.columns.tolist()}"
            )

        return df

Column inference

In [8]:
def infer_embedding_columns(df: pd.DataFrame, exclude_cols: List[str]) -> List[str]:
    return [c for c in df.columns if c not in exclude_cols]


def infer_image_tabular_cols(img_df: pd.DataFrame, tab_df: pd.DataFrame, cfg: Config):
    img_exclude = {
        cfg.image_id_col,
        cfg.patient_id_col,
        cfg.image_index_col,
        cfg.label_col,
        cfg.split_col,
        cfg.gest_week_col,
        "image_path",
        "filename",
        "class_name",
        "fold",
        "ok"
    }

    tab_exclude = {
        cfg.patient_id_col,
        cfg.label_col,
        cfg.split_col
    }

    img_cols = [
        c for c in img_df.columns
        if c not in img_exclude and pd.api.types.is_numeric_dtype(img_df[c])
    ]

    tab_cols = [
        c for c in tab_df.columns
        if c not in tab_exclude and pd.api.types.is_numeric_dtype(tab_df[c])
    ]

    return img_cols, tab_cols

ImageSelector

In [9]:
class ImageSelector:
    def __init__(self, config: Config):
        self.cfg = config

    def _evenly_spaced_indices(self, n: int, k: int) -> List[int]:
        if n <= k:
            return list(range(n))
        raw = np.linspace(0, n - 1, num=k)
        idx = np.round(raw).astype(int).tolist()
        idx = sorted(list(dict.fromkeys(idx)))
        while len(idx) < k:
            for i in range(n):
                if i not in idx:
                    idx.append(i)
                if len(idx) == k:
                    break
        return sorted(idx[:k])

    def _select_evenly_spaced(self, df_patient: pd.DataFrame) -> pd.DataFrame:
        df_patient = df_patient.sort_values(
            by=self.cfg.image_index_col
        ).reset_index(drop=True)
        idx = self._evenly_spaced_indices(
            len(df_patient), self.cfg.max_images_per_patient
        )
        return df_patient.iloc[idx].copy()

    def _select_gestational_spread(self, df_patient: pd.DataFrame) -> pd.DataFrame:
        df_patient = df_patient.copy()
        if self.cfg.gest_week_col not in df_patient.columns:
            return self._select_evenly_spaced(df_patient)
        df_patient["gest_week_clean"] = pd.to_numeric(
            df_patient[self.cfg.gest_week_col], errors="coerce"
        )
        df_patient.loc[
            df_patient["gest_week_clean"] < 0, "gest_week_clean"
        ] = np.nan
        valid_df = (
            df_patient[df_patient["gest_week_clean"].notna()]
            .sort_values("gest_week_clean")
            .reset_index(drop=True)
        )
        unknown_df = (
            df_patient[df_patient["gest_week_clean"].isna()]
            .sort_values(self.cfg.image_index_col)
            .reset_index(drop=True)
        )
        selected_parts = []
        if len(valid_df) >= self.cfg.max_images_per_patient:
            idx = self._evenly_spaced_indices(
                len(valid_df), self.cfg.max_images_per_patient
            )
            selected_parts.append(valid_df.iloc[idx].copy())
        else:
            if len(valid_df) > 0:
                selected_parts.append(valid_df.copy())
            remaining = self.cfg.max_images_per_patient - len(valid_df)
            if remaining > 0 and len(unknown_df) > 0:
                idx = self._evenly_spaced_indices(len(unknown_df), remaining)
                selected_parts.append(unknown_df.iloc[idx].copy())
        if len(selected_parts) == 0:
            return self._select_evenly_spaced(df_patient)
        out = pd.concat(selected_parts, axis=0).drop_duplicates()
        out = out.sort_values(
            by=self.cfg.image_index_col
        ).reset_index(drop=True)
        if len(out) > self.cfg.max_images_per_patient:
            out = out.iloc[:self.cfg.max_images_per_patient].copy()
        return out

    def select_image_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        # FIXED: if max_images is None or strategy is "all", always return everything
        if (
            self.cfg.max_images_per_patient is None
            or self.cfg.selection_strategy == "all"
        ):
            return df.copy()

        selected = []
        for _, df_patient in df.groupby(self.cfg.patient_id_col):
            if len(df_patient) <= self.cfg.max_images_per_patient:
                selected.append(df_patient.copy())
            elif self.cfg.selection_strategy == "evenly_spaced":
                selected.append(self._select_evenly_spaced(df_patient))
            elif self.cfg.selection_strategy == "gestational_spread":
                selected.append(self._select_gestational_spread(df_patient))
            else:
                raise ValueError(
                    f"Unknown selection strategy: {self.cfg.selection_strategy}"
                )
        return pd.concat(selected, axis=0).reset_index(drop=True)

MultimodalDatasetBuilder

In [10]:
class MultimodalDatasetBuilder:
    def __init__(self, config: Config, path_manager, embedding_loader, image_selector):
        self.cfg = config
        self.path_manager = path_manager
        self.embedding_loader = embedding_loader
        self.image_selector = image_selector

    def _prepare_image_df(self, img_df: pd.DataFrame) -> pd.DataFrame:
        if self.cfg.image_id_col not in img_df.columns:
            raise ValueError(f"Missing required image id column: {self.cfg.image_id_col}")

        if self.cfg.patient_id_col not in img_df.columns:
            parsed = img_df[self.cfg.image_id_col].apply(parse_image_id)
            img_df[self.cfg.image_index_col] = [x[0] for x in parsed]
            img_df[self.cfg.patient_id_col] = [x[1] for x in parsed]
        else:
            if self.cfg.image_index_col not in img_df.columns:
                parsed = img_df[self.cfg.image_id_col].apply(parse_image_id)
                img_df[self.cfg.image_index_col] = [x[0] for x in parsed]

        if self.cfg.split_col not in img_df.columns:
            raise ValueError(
                f"Expected '{self.cfg.split_col}' in image embeddings, but it was not found. "
                f"Available columns: {img_df.columns.tolist()}"
            )

        if self.cfg.label_col not in img_df.columns:
            raise ValueError(
                f"Expected '{self.cfg.label_col}' in image embeddings, but it was not found. "
                f"Available columns: {img_df.columns.tolist()}"
            )

        return img_df

    def build_fold_image_row_dataframe(self, fold_idx: int):
        self.path_manager.validate_paths(fold_idx)

        img_df = self.embedding_loader.load_image_embeddings(fold_idx)
        tab_df = self.embedding_loader.load_tabular_embeddings(fold_idx)

        img_df = self._prepare_image_df(img_df)
        img_cols, tab_cols = infer_image_tabular_cols(img_df, tab_df, self.cfg)

        tab_keep_cols = [self.cfg.patient_id_col] + tab_cols
        if self.cfg.label_col in tab_df.columns and self.cfg.label_col not in tab_keep_cols:
            tab_keep_cols.append(self.cfg.label_col)

        merged = img_df.merge(
            tab_df[tab_keep_cols].drop_duplicates(subset=[self.cfg.patient_id_col]),
            on=self.cfg.patient_id_col,
            how="inner",
            suffixes=("", "_tab")
        )

        self.cfg.img_dim = len(img_cols)
        self.cfg.tab_dim = len(tab_cols)

        return merged.reset_index(drop=True), img_cols, tab_cols

    def split_image_row_dataframe(self, df: pd.DataFrame):
        if "train" not in df[self.cfg.split_col].unique():
            raise ValueError("Expected 'train' in split column.")
        if "test" not in df[self.cfg.split_col].unique():
            raise ValueError("Expected 'test' in split column.")

        train_full_df = df[df[self.cfg.split_col] == "train"].copy()
        test_df = df[df[self.cfg.split_col] == "test"].copy()

        patient_df = train_full_df[[self.cfg.patient_id_col, self.cfg.label_col]].drop_duplicates()

        train_patients, val_patients = train_test_split(
            patient_df,
            test_size=self.cfg.val_size,
            stratify=patient_df[self.cfg.label_col],
            random_state=self.cfg.random_state
        )

        train_patient_ids = set(train_patients[self.cfg.patient_id_col])
        val_patient_ids = set(val_patients[self.cfg.patient_id_col])

        train_df = train_full_df[train_full_df[self.cfg.patient_id_col].isin(train_patient_ids)].copy()
        val_df = train_full_df[train_full_df[self.cfg.patient_id_col].isin(val_patient_ids)].copy()

        return train_df, val_df, test_df

PatientSequenceBuilder

In [11]:
class PatientSequenceBuilder:
    def __init__(self, config: Config, image_selector: ImageSelector):
        self.cfg = config
        self.image_selector = image_selector

    def build_patient_level_dataframe(
        self,
        image_row_df: pd.DataFrame,
        img_cols: List[str],
        tab_cols: List[str]
    ) -> pd.DataFrame:
        rows = []

        for patient_id, df_patient in image_row_df.groupby(self.cfg.patient_id_col):
            df_patient = df_patient.copy().reset_index(drop=True)
            df_patient_selected = self.image_selector.select_image_rows(df_patient)

            label = int(df_patient_selected[self.cfg.label_col].iloc[0])
            split = df_patient_selected[self.cfg.split_col].iloc[0]

            tab_vector = df_patient_selected.iloc[0][tab_cols].astype(np.float32).values
            image_vectors = df_patient_selected[img_cols].astype(np.float32).values.tolist()
            image_ids = df_patient_selected[self.cfg.image_id_col].astype(str).tolist()

            patient_row = {
                self.cfg.patient_id_col: patient_id,
                self.cfg.label_col: label,
                self.cfg.split_col: split,
                "image_vectors": image_vectors,
                "image_ids": image_ids,
                "num_images_used": len(image_vectors),
                "tab_vector": tab_vector
            }

            rows.append(patient_row)

        return pd.DataFrame(rows)

Dataset + DataLoaderFactory

In [12]:
# ------------------------------------------------------------------
# 3. PatientSequenceDataset
# Changes: max_k is computed from the actual dataset at __init__ time
#          so it never crashes when max_images_per_patient is None
# ------------------------------------------------------------------
class PatientSequenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, config: Config):
        self.df  = df.reset_index(drop=True)
        self.cfg = config

        # FIXED: derive true max sequence length from this dataset's data,
        # not from cfg.max_images_per_patient which may be None
        self.max_k = int(
            self.df["num_images_used"].max()
        ) if len(self.df) > 0 else 1
        print(f"  PatientSequenceDataset: {len(self.df)} patients, "
              f"max_images_used={self.max_k}")

    def __len__(self):
        return len(self.df)

    def _pad_image_sequence(self, image_vectors: List[List[float]]):
        # FIXED: use self.max_k (from data) instead of cfg.max_images_per_patient
        img_dim = self.cfg.img_dim
        seq = np.zeros((self.max_k, img_dim), dtype=np.float32)
        key_padding_mask = np.ones((self.max_k,), dtype=bool)  # True = padding

        k = min(len(image_vectors), self.max_k)
        if k > 0:
            seq[:k] = np.array(image_vectors[:k], dtype=np.float32)
            key_padding_mask[:k] = False
        return seq, key_padding_mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_seq, img_key_padding_mask = self._pad_image_sequence(
            row["image_vectors"]
        )
        tab_vector = np.array(row["tab_vector"], dtype=np.float32)
        label      = np.float32(row[self.cfg.label_col])

        return {
            "img_seq":              torch.tensor(img_seq,              dtype=torch.float32),
            "img_key_padding_mask": torch.tensor(img_key_padding_mask, dtype=torch.bool),
            "tab_emb":              torch.tensor(tab_vector,           dtype=torch.float32),
            "label":                torch.tensor(label,                dtype=torch.float32),
            "patient_id":           str(row[self.cfg.patient_id_col]),
            "num_images_used":      int(row["num_images_used"])
        }


# ------------------------------------------------------------------
# 4. PatientSequenceDataLoaderFactory
# Changes: computes global max_images across ALL three splits,
#          stores it back to cfg so CrossAttentionPatientModel's
#          padding is consistent across train / val / test batches
# ------------------------------------------------------------------
class PatientSequenceDataLoaderFactory:
    def __init__(self, config: Config):
        self.cfg = config

    def build(
        self,
        train_df: pd.DataFrame,
        val_df:   pd.DataFrame,
        test_df:  pd.DataFrame
    ):
        # FIXED: compute the true global max across all splits before
        # building any Dataset so every dataset pads to the same length
        global_max_k = int(
            pd.concat([train_df, val_df, test_df])["num_images_used"].max()
        )
        print(f"Global max images per patient across splits: {global_max_k}")

        # Write back into cfg so it is available to models / datasets
        self.cfg.max_images_per_patient = global_max_k

        train_ds = PatientSequenceDataset(train_df, self.cfg)
        val_ds   = PatientSequenceDataset(val_df,   self.cfg)
        test_ds  = PatientSequenceDataset(test_df,  self.cfg)

        train_loader = DataLoader(
            train_ds, batch_size=self.cfg.batch_size, shuffle=True
        )
        val_loader   = DataLoader(
            val_ds, batch_size=self.cfg.batch_size, shuffle=False
        )
        test_loader  = DataLoader(
            test_ds, batch_size=self.cfg.batch_size, shuffle=False
        )
        return train_loader, val_loader, test_loader


Models

In [13]:
class MLPPatientModel(nn.Module):
    def __init__(self, img_dim: int, tab_dim: int, hidden_dim: int, dropout: float):
        super().__init__()

        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, img_seq, img_key_padding_mask, tab_emb):
        valid_mask = (~img_key_padding_mask).float().unsqueeze(-1)
        pooled_img = (img_seq * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1.0)

        img_feat = self.img_proj(pooled_img)
        tab_feat = self.tab_proj(tab_emb)

        fused = torch.cat([img_feat, tab_feat], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits


class CrossAttentionPatientModel(nn.Module):
    def __init__(
        self,
        img_dim: int,
        tab_dim: int,
        embed_dim: int,
        num_heads: int,
        num_layers: int,
        dropout: float,
        classifier_hidden_dim: int
    ):
        super().__init__()

        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 2,
            dropout=dropout,
            batch_first=True
        )
        self.image_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.tab_to_img_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.img_to_tab_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.img_norm = nn.LayerNorm(embed_dim)
        self.tab_norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim // 2, 1)
        )

    def masked_mean_pool(self, x: torch.Tensor, key_padding_mask: torch.Tensor):
        valid_mask = (~key_padding_mask).float().unsqueeze(-1)
        x_sum = (x * valid_mask).sum(dim=1)
        denom = valid_mask.sum(dim=1).clamp(min=1.0)
        return x_sum / denom

    def forward(self, img_seq, img_key_padding_mask, tab_emb):
        img_tokens = self.img_proj(img_seq)                      # [B, K, E]
        tab_token = self.tab_proj(tab_emb).unsqueeze(1)         # [B, 1, E]

        img_tokens = self.image_encoder(
            img_tokens,
            src_key_padding_mask=img_key_padding_mask
        )

        tab_refined, _ = self.tab_to_img_attn(
            query=tab_token,
            key=img_tokens,
            value=img_tokens,
            key_padding_mask=img_key_padding_mask
        )
        tab_refined = self.tab_norm(tab_refined)

        img_refined, _ = self.img_to_tab_attn(
            query=img_tokens,
            key=tab_refined,
            value=tab_refined
        )
        img_refined = self.img_norm(img_refined)

        pooled_img = self.masked_mean_pool(img_refined, img_key_padding_mask)
        pooled_tab = tab_refined.squeeze(1)

        fused = torch.cat([pooled_img, pooled_tab], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits


class ModelFactory:
    def __init__(self, config: Config):
        self.cfg = config

    def build(self):
        if self.cfg.fusion_model_name == "mlp_patient":
            return MLPPatientModel(
                img_dim=self.cfg.img_dim,
                tab_dim=self.cfg.tab_dim,
                hidden_dim=self.cfg.hidden_dim,
                dropout=self.cfg.dropout
            )

        if self.cfg.fusion_model_name == "cross_attention_patient":
            return CrossAttentionPatientModel(
                img_dim=self.cfg.img_dim,
                tab_dim=self.cfg.tab_dim,
                embed_dim=self.cfg.embed_dim,
                num_heads=self.cfg.num_heads,
                num_layers=self.cfg.num_layers,
                dropout=self.cfg.dropout,
                classifier_hidden_dim=self.cfg.classifier_hidden_dim
            )

        raise ValueError(f"Unsupported fusion model: {self.cfg.fusion_model_name}")

LossManager

In [14]:
class BCELossWrapper(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return F.binary_cross_entropy_with_logits(
            input=logits,
            target=targets,
            reduction="none"
        )


class WeightedBCELossWrapper(nn.Module):
    def __init__(self, pos_weight: float):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        pos_weight_tensor = torch.tensor(
            self.pos_weight,
            dtype=logits.dtype,
            device=logits.device
        )

        return F.binary_cross_entropy_with_logits(
            input=logits,
            target=targets,
            reduction="none",
            pos_weight=pos_weight_tensor
        )


class FocalLossWrapper(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss = F.binary_cross_entropy_with_logits(
            input=logits,
            target=targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)

        focal_loss = alpha_t * ((1.0 - pt) ** self.gamma) * bce_loss
        return focal_loss


class LossManager:
    def __init__(self, config: Config, pos_weight: Optional[float] = None):
        self.cfg = config
        self.pos_weight = pos_weight
        self.loss_fn = self._build_loss_function()

    def _build_loss_function(self):
        if self.cfg.loss_name == "bce":
            return BCELossWrapper()

        if self.cfg.loss_name == "weighted_bce":
            if self.pos_weight is None:
                raise ValueError("pos_weight must be provided when loss_name='weighted_bce'.")
            return WeightedBCELossWrapper(pos_weight=self.pos_weight)

        if self.cfg.loss_name == "focal":
            return FocalLossWrapper(
                alpha=self.cfg.focal_alpha,
                gamma=self.cfg.focal_gamma
            )

        raise ValueError(f"Unsupported loss_name: {self.cfg.loss_name}")

    def __call__(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        per_sample_loss = self.loss_fn(logits, targets)
        return per_sample_loss.mean()

PatientLevelEvaluator

Trainer

In [15]:
class Trainer:
    def __init__(
        self,
        config: Config,
        model,
        loss_manager,
        optimizer,
        train_loader,
        val_loader,
        test_loader,
        fold_idx: int,
        checkpoint_dir: str
    ):
        self.cfg = config
        self.model = model.to(self.cfg.device)
        self.loss_manager = loss_manager
        self.optimizer = optimizer

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.fold_idx = fold_idx

        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.best_metric_name = self.cfg.best_metric
        self.best_metric_value = -np.inf
        self.best_epoch = None
        self.best_state = None
        self.best_checkpoint_path = self.checkpoint_dir / f"best_model_fold_{self.fold_idx}.pt"
        self.best_val_metrics = None

        self.history = []

    def _move_batch(self, batch):
        img_seq = batch["img_seq"].to(self.cfg.device)
        img_key_padding_mask = batch["img_key_padding_mask"].to(self.cfg.device)
        tab_emb = batch["tab_emb"].to(self.cfg.device)
        labels = batch["label"].to(self.cfg.device)
        patient_ids = batch["patient_id"]
        return img_seq, img_key_padding_mask, tab_emb, labels, patient_ids

    def train_one_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_batches = 0

        for batch in self.train_loader:
            img_seq, img_key_padding_mask, tab_emb, labels, _ = self._move_batch(batch)

            self.optimizer.zero_grad()

            logits = self.model(
                img_seq=img_seq,
                img_key_padding_mask=img_key_padding_mask,
                tab_emb=tab_emb
            )

            loss = self.loss_manager(logits, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            total_batches += 1

        return total_loss / max(total_batches, 1)

    @torch.no_grad()
    def evaluate_loader(self, loader):
        self.model.eval()

        total_loss = 0.0
        total_batches = 0

        all_probs = []
        all_labels = []

        for batch in loader:
            img_seq, img_key_padding_mask, tab_emb, labels, _ = self._move_batch(batch)

            logits = self.model(
                img_seq=img_seq,
                img_key_padding_mask=img_key_padding_mask,
                tab_emb=tab_emb
            )

            loss = self.loss_manager(logits, labels)

            probs = torch.sigmoid(logits).cpu().numpy()
            labels_np = labels.cpu().numpy()

            total_loss += loss.item()
            total_batches += 1

            all_probs.extend(probs.tolist())
            all_labels.extend(labels_np.tolist())

        avg_loss = total_loss / max(total_batches, 1)

        metrics = compute_binary_metrics(
            y_true=np.array(all_labels),
            y_prob=np.array(all_probs),
            threshold=self.cfg.decision_threshold
        )
        metrics["loss"] = avg_loss

        return metrics, np.array(all_labels), np.array(all_probs)

    def _save_best_checkpoint(self):
        torch.save(
            {
                "fold": self.fold_idx,
                "best_epoch": self.best_epoch,
                "best_metric_name": self.best_metric_name,
                "best_metric_value": self.best_metric_value,
                "best_val_metrics": self.best_val_metrics,
                "model_state_dict": self.best_state,
                "config": asdict(self.cfg),
            },
            self.best_checkpoint_path
        )

    def fit(self):
        patience_counter = 0

        for epoch in range(1, self.cfg.max_epochs + 1):
            train_loss = self.train_one_epoch()
            val_metrics, _, _ = self.evaluate_loader(self.val_loader)

            row = {
                "fold": self.fold_idx,
                "epoch": epoch,
                "train_loss": train_loss,
                **{f"val_{k}": v for k, v in val_metrics.items()}
            }
            self.history.append(row)

            print(
                f"Fold {self.fold_idx} | Epoch {epoch:03d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | "
                f"val_recall={val_metrics['recall']:.4f} | "
                f"val_f1={val_metrics['f1']:.4f} | "
                f"val_pr_auc={val_metrics['pr_auc']:.4f}"
            )

            current_metric = val_metrics[self.best_metric_name]

            if current_metric > self.best_metric_value:
                self.best_metric_value = current_metric
                self.best_epoch = epoch
                self.best_val_metrics = val_metrics.copy()
                self.best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in self.model.state_dict().items()
                }
                self._save_best_checkpoint()
                patience_counter = 0

                print(
                    f"  ✓ New best val_{self.best_metric_name}={current_metric:.4f} "
                    f"— checkpoint saved"
                )
            else:
                patience_counter += 1

            if patience_counter >= self.cfg.patience:
                print(
                    f"Early stopping fold {self.fold_idx} | "
                    f"best val_{self.best_metric_name}={self.best_metric_value:.4f}"
                )
                break

        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)

    def test(self):
        return self.evaluate_loader(self.test_loader)

    def get_history_df(self):
        return pd.DataFrame(self.history)

    def get_best_model_info(self):
        return {
            "best_epoch": self.best_epoch,
            "best_metric_name": self.best_metric_name,
            "best_metric_value": self.best_metric_value,
            "best_val_metrics": self.best_val_metrics,
            "checkpoint_path": str(self.best_checkpoint_path),
        }

FoldRunner

In [16]:
class FoldRunner:
    def __init__(self, config: Config):
        self.cfg = config
        self.path_manager        = PathManager(config)
        self.embedding_loader    = EmbeddingLoader(config, self.path_manager)
        self.image_selector      = ImageSelector(config)
        self.dataset_builder     = MultimodalDatasetBuilder(
            config          = config,
            path_manager    = self.path_manager,
            embedding_loader= self.embedding_loader,
            image_selector  = self.image_selector
        )
        self.patient_sequence_builder = PatientSequenceBuilder(
            config, self.image_selector
        )
        self.dataloader_factory = PatientSequenceDataLoaderFactory(config)

    def _compute_pos_weight(self, train_df: pd.DataFrame):
        if self.cfg.loss_name != "weighted_bce":
            return None
        if self.cfg.pos_weight is not None:
            return self.cfg.pos_weight
        pos_count = train_df[self.cfg.label_col].sum()
        neg_count = len(train_df) - pos_count
        return float(neg_count / max(pos_count, 1.0))

    def _verify_patient_image_mapping(
        self, merged_df: pd.DataFrame, img_cols: List[str], tab_cols: List[str]
    ):
        """
        Print a quick sanity-check so you can confirm the join is correct:
        - How many patients matched between image and tabular files
        - Image distribution per patient (shows we are using ALL images now)
        - Label consistency between image-side and tabular-side labels
        """
        n_patients    = merged_df[self.cfg.patient_id_col].nunique()
        imgs_per_pat  = merged_df.groupby(
            self.cfg.patient_id_col
        ).size()
        label_mismatch = 0
        if "label_tab" in merged_df.columns:
            label_mismatch = (
                merged_df[self.cfg.label_col] != merged_df["label_tab"]
            ).sum()

        print(f"\n  --- Mapping verification ---")
        print(f"  Matched patients (image ∩ tabular): {n_patients}")
        print(f"  Images per patient — "
              f"min={imgs_per_pat.min()}, "
              f"median={imgs_per_pat.median():.0f}, "
              f"max={imgs_per_pat.max()}, "
              f"mean={imgs_per_pat.mean():.1f}")
        print(f"  Label mismatches (image vs tabular label col): {label_mismatch}")
        print(f"  img_dim={len(img_cols)}, tab_dim={len(tab_cols)}")
        print(f"  ----------------------------\n")

    def run_fold(self, fold_idx: int):
      print(f"\n========== Fold {fold_idx} ==========")

      image_row_df, img_cols, tab_cols = self.dataset_builder.build_fold_image_row_dataframe(fold_idx)
      train_rows_df, val_rows_df, test_rows_df = self.dataset_builder.split_image_row_dataframe(image_row_df)

      train_patient_df = self.patient_sequence_builder.build_patient_level_dataframe(train_rows_df, img_cols, tab_cols)
      val_patient_df = self.patient_sequence_builder.build_patient_level_dataframe(val_rows_df, img_cols, tab_cols)
      test_patient_df = self.patient_sequence_builder.build_patient_level_dataframe(test_rows_df, img_cols, tab_cols)

      print("Train patients:", len(train_patient_df))
      print("Val patients:", len(val_patient_df))
      print("Test patients:", len(test_patient_df))

      train_loader, val_loader, test_loader = self.dataloader_factory.build(
          train_patient_df, val_patient_df, test_patient_df
      )

      model = ModelFactory(self.cfg).build()
      pos_weight = self._compute_pos_weight(train_patient_df)
      loss_manager = LossManager(self.cfg, pos_weight=pos_weight)

      optimizer = torch.optim.AdamW(
          model.parameters(),
          lr=self.cfg.lr,
          weight_decay=self.cfg.weight_decay
      )

      checkpoint_dir = Path(self.cfg.output_root) / "checkpoints"

      trainer = Trainer(
          config=self.cfg,
          model=model,
          loss_manager=loss_manager,
          optimizer=optimizer,
          train_loader=train_loader,
          val_loader=val_loader,
          test_loader=test_loader,
          fold_idx=fold_idx,
          checkpoint_dir=str(checkpoint_dir)
      )

      trainer.fit()
      test_metrics, y_true, y_prob = trainer.test()
      best_model_info = trainer.get_best_model_info()

      return {
        "fold": fold_idx,
        "test_metrics": test_metrics,
        "history_df": trainer.get_history_df(),
        "y_true_patient": y_true,
        "y_prob_patient": y_prob,
        "n_train_patients": len(train_patient_df),
        "n_val_patients": len(val_patient_df),
        "n_test_patients": len(test_patient_df),
        "best_model_info": best_model_info,
      }

CrossFoldExperiment

In [17]:
class CrossFoldExperiment:
    def __init__(self, config: Config):
        self.cfg = config
        self.fold_runner = FoldRunner(config)
        self.fold_results = []
        self.history_frames = []

    def run(self):
        for fold_idx in range(1, self.cfg.n_folds + 1):
            result = self.fold_runner.run_fold(fold_idx)
            self.fold_results.append(result)
            self.history_frames.append(result["history_df"])

    def get_metrics_df(self):
        rows = []
        for result in self.fold_results:
            row = {
                "fold": result["fold"],
                **result["test_metrics"],
                "n_train_patients": result["n_train_patients"],
                "n_val_patients": result["n_val_patients"],
                "n_test_patients": result["n_test_patients"],
                "best_epoch": result["best_model_info"]["best_epoch"],
                "best_val_metric": result["best_model_info"]["best_metric_value"],
                "checkpoint_path": result["best_model_info"]["checkpoint_path"],
            }
            rows.append(row)
        return pd.DataFrame(rows)

    def get_history_df(self):
        if len(self.history_frames) == 0:
            return pd.DataFrame()
        return pd.concat(self.history_frames, axis=0, ignore_index=True)

    def summarize(self):
        metrics_df = self.get_metrics_df()
        summary_df = metrics_df[["precision", "recall", "f1", "accuracy", "auc", "pr_auc", "loss"]].agg(["mean", "std"]).T
        summary_df = summary_df.reset_index()
        summary_df.columns = ["metric", "mean", "std"]
        return metrics_df, summary_df

    def get_best_fold_model(self):
        best_result = max(
            self.fold_results,
            key=lambda r: r["best_model_info"]["best_metric_value"]
        )
        return best_result["best_model_info"]

ResultSaver

In [18]:
class ResultSaver:
    def __init__(self, config: Config):
        self.cfg = config
        self.output_dir = Path(self.cfg.output_root)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def save(self, metrics_df: pd.DataFrame, summary_df: pd.DataFrame, history_df: pd.DataFrame):
        metrics_df.to_csv(self.output_dir / "fold_test_metrics.csv", index=False)
        summary_df.to_csv(self.output_dir / "summary_metrics.csv", index=False)
        history_df.to_csv(self.output_dir / "training_history.csv", index=False)

        with open(self.output_dir / "config.json", "w") as f:
            json.dump(asdict(self.cfg), f, indent=2)

        print("Saved outputs to:", self.output_dir)



```
# This is formatted as code
```



experiment

In [19]:
experiment = CrossFoldExperiment(CFG)
experiment.run()

metrics_df, summary_df = experiment.summarize()
history_df = experiment.get_history_df()

print("=== Fold-wise Metrics ===")
print(metrics_df)

print("\n=== Summary ===")
print(summary_df)


========== Fold 1 ==========
Image path: /content/gdrive/MyDrive/CARDIUM/CARDIUM_image_embeddings/cardium_fetalclip_embeddings_fold_1.parquet
Exists? True
Tabular path: /content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_fold0.parquet
Exists? True
Loaded image embeddings: (6558, 777)
Image columns: ['image_path', 'fold', 'split', 'class_name', 'label', 'patient_id', 'image_id', 'filename', 'ok', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb_55', 'emb_56', 'emb_5

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Fold 1 | Epoch 001 | train_loss=1.2798 | val_loss=1.2141 | val_recall=0.7143 | val_f1=0.1786 | val_pr_auc=0.0893
  ✓ New best val_pr_auc=0.0893 — checkpoint saved
Fold 1 | Epoch 002 | train_loss=1.2176 | val_loss=1.1252 | val_recall=0.8571 | val_f1=0.1846 | val_pr_auc=0.1598
  ✓ New best val_pr_auc=0.1598 — checkpoint saved
Fold 1 | Epoch 003 | train_loss=1.1171 | val_loss=1.1380 | val_recall=0.7143 | val_f1=0.2000 | val_pr_auc=0.1223
Fold 1 | Epoch 004 | train_loss=1.0778 | val_loss=1.1697 | val_recall=0.2857 | val_f1=0.2000 | val_pr_auc=0.1658
  ✓ New best val_pr_auc=0.1658 — checkpoint saved
Fold 1 | Epoch 005 | train_loss=1.0343 | val_loss=1.2529 | val_recall=1.0000 | val_f1=0.1556 | val_pr_auc=0.2209
  ✓ New best val_pr_auc=0.2209 — checkpoint saved
Fold 1 | Epoch 006 | train_loss=1.0528 | val_loss=0.9580 | val_recall=0.7143 | val_f1=0.2941 | val_pr_auc=0.6931
  ✓ New best val_pr_auc=0.6931 — checkpoint saved
Fold 1 | Epoch 007 | train_loss=0.7724 | val_loss=0.9319 | val_recall=0.

Save

In [20]:
saver = ResultSaver(CFG)
saver.save(metrics_df, summary_df, history_df)

Saved outputs to: /content/gdrive/MyDrive/CARDIUM/outputs_multimodal


Comparison

In [21]:
# ============================================================
# EXPERIMENT RUNNER
# Compares BCE vs Weighted BCE vs Focal Loss
# Primary metric: recall (minimize missed CHD cases)
# Drop this cell at the bottom of the multimodal notebook
# and run it — everything above must already be defined
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy
from dataclasses import asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# EXPERIMENT CONFIGS
# One Config object per experiment — nothing shared between runs
# ============================================================

def make_bce_config():
    cfg = Config()
    cfg.loss_name   = "bce"
    cfg.pos_weight  = None
    cfg.output_root = "/content/gdrive/MyDrive/CARDIUM/outputs_bce"
    return cfg


def make_weighted_bce_config():
    cfg = Config()
    cfg.loss_name  = "weighted_bce"
    # pos_weight=None means FoldRunner computes it from data as neg/pos ratio
    # for CARDIUM that is ~1029/74 ≈ 13.9 — heavily penalises missing a CHD case
    cfg.pos_weight  = None
    cfg.output_root = "/content/gdrive/MyDrive/CARDIUM/outputs_weighted_bce"
    return cfg


def make_focal_config():
    cfg = Config()
    cfg.loss_name   = "focal"
    # alpha=0.75 upweights the positive (CHD) class
    # gamma=2.0 concentrates learning on hard examples
    cfg.focal_alpha = 0.75
    cfg.focal_gamma = 2.0
    cfg.output_root = "/content/gdrive/MyDrive/CARDIUM/outputs_focal"
    return cfg


EXPERIMENT_CONFIGS = {
    "bce":          make_bce_config(),
    "weighted_bce": make_weighted_bce_config(),
    "focal":        make_focal_config(),
}


# ============================================================
# SINGLE EXPERIMENT RUNNER
# Runs 3-fold CV for one loss config and saves all outputs
# ============================================================

def run_single_experiment(name: str, cfg: Config) -> dict:
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {name.upper()}")
    print(f"  loss={cfg.loss_name}  "
          f"focal_alpha={getattr(cfg,'focal_alpha','-')}  "
          f"focal_gamma={getattr(cfg,'focal_gamma','-')}  "
          f"pos_weight={cfg.pos_weight or 'auto'}")
    print(f"{'='*60}")

    Path(cfg.output_root).mkdir(parents=True, exist_ok=True)

    experiment = CrossFoldExperiment(cfg)
    experiment.run()
    metrics_df, summary_df = experiment.summarize()
    history_df = experiment.get_history_df()

    saver = ResultSaver(cfg)
    saver.save(metrics_df, summary_df, history_df)

    return {
        "name":       name,
        "metrics_df": metrics_df,
        "summary_df": summary_df,
        "history_df": history_df,
    }


# ============================================================
# COMPARISON TABLE
# Focuses on the clinical objective:
#   primary  → recall   (miss as few CHD cases as possible)
#   secondary → precision (false positive burden on the system)
#   tertiary → f1, pr_auc, auc
# ============================================================

METRICS_TO_SHOW = ["recall", "precision", "f1", "pr_auc", "auc"]


def build_comparison_table(all_results: dict) -> pd.DataFrame:
    rows = []
    for name, result in all_results.items():
        s = result["summary_df"]
        s = s.set_index("metric")
        row = {"experiment": name}
        for m in METRICS_TO_SHOW:
            if m in s.index:
                mean = s.loc[m, "mean"]
                std  = s.loc[m, "std"]
                row[m] = f"{mean:.3f} ± {std:.3f}"
            else:
                row[m] = "—"
        rows.append(row)
    return pd.DataFrame(rows).set_index("experiment")


def print_comparison(table: pd.DataFrame):
    print("\n")
    print("=" * 72)
    print("  EXPERIMENT COMPARISON  (mean ± std across 3 folds)")
    print("  Primary objective: RECALL — minimize missed CHD cases")
    print("=" * 72)
    print(table.to_string())
    print("=" * 72)

    # identify best recall
    best_exp = None
    best_val = -1.0
    for exp in table.index:
        val_str = table.loc[exp, "recall"]
        if val_str == "—":
            continue
        val = float(val_str.split(" ")[0])
        if val > best_val:
            best_val = val
            best_exp = exp

    if best_exp:
        print(f"\n  Best recall: {best_exp}  ({best_val:.3f})")

    # interpret the comparison
    print("\n  Interpretation guide:")
    print("  · recall    ↑ = fewer missed CHD cases (primary goal)")
    print("  · precision ↑ = fewer false alarms sent to specialists")
    print("  · pr_auc    ↑ = overall tradeoff quality (imbalanced metric)")
    print("  · f1        ↑ = harmonic mean (balanced view)")
    print("=" * 72)


def save_comparison_table(table: pd.DataFrame, out_dir: str):
    path = Path(out_dir) / "experiment_comparison.csv"
    table.to_csv(path)
    print(f"\nComparison table saved to: {path}")


# ============================================================
# PER-FOLD DETAIL TABLE
# Shows variance — important for a dataset this small
# ============================================================

def build_fold_detail_table(all_results: dict) -> pd.DataFrame:
    rows = []
    for name, result in all_results.items():
        mdf = result["metrics_df"]
        for _, row in mdf.iterrows():
            rows.append({
                "experiment": name,
                "fold":       int(row["fold"]),
                "recall":     round(row["recall"],    3),
                "precision":  round(row["precision"], 3),
                "f1":         round(row["f1"],        3),
                "pr_auc":     round(row.get("pr_auc", float("nan")), 3),
                "auc":        round(row.get("auc",    float("nan")), 3),
            })
    return pd.DataFrame(rows)


def print_fold_detail(detail: pd.DataFrame):
    print("\n")
    print("=" * 72)
    print("  PER-FOLD RESULTS (shows variance across folds)")
    print("=" * 72)
    print(detail.to_string(index=False))
    print("=" * 72)


# ============================================================
# TRAINING CURVE SUMMARY
# Quick check that all three models actually learned
# (if val_pr_auc never rises above ~0.1 the embeddings are broken)
# ============================================================

def print_training_summary(all_results: dict):
    print("\n")
    print("=" * 72)
    print("  TRAINING CURVE SUMMARY")
    print("  val_pr_auc at best epoch per fold (should be > 0.15 to confirm learning)")
    print("=" * 72)
    for name, result in all_results.items():
        hdf = result["history_df"]
        if "val_pr_auc" not in hdf.columns:
            print(f"  {name}: val_pr_auc column not found")
            continue
        for fold_idx in sorted(hdf["fold"].unique()):
            fold_h = hdf[hdf["fold"] == fold_idx]
            best   = fold_h["val_pr_auc"].max()
            epochs = len(fold_h)
            print(f"  {name:<15}  fold {fold_idx}  "
                  f"best val_pr_auc={best:.4f}  epochs run={epochs}")
    print("=" * 72)


# ============================================================
# MAIN — run all three experiments then compare
# ============================================================

COMPARISON_OUT_DIR = "/content/gdrive/MyDrive/CARDIUM/outputs_comparison"
Path(COMPARISON_OUT_DIR).mkdir(parents=True, exist_ok=True)

all_results = {}
for exp_name, exp_cfg in EXPERIMENT_CONFIGS.items():
    all_results[exp_name] = run_single_experiment(exp_name, exp_cfg)

# Build and print comparison
comparison_table = build_comparison_table(all_results)
print_comparison(comparison_table)
save_comparison_table(comparison_table, COMPARISON_OUT_DIR)

# Per-fold detail
fold_detail = build_fold_detail_table(all_results)
print_fold_detail(fold_detail)
fold_detail.to_csv(Path(COMPARISON_OUT_DIR) / "fold_detail.csv", index=False)

# Training curve check
print_training_summary(all_results)

print("\nAll outputs saved to:", COMPARISON_OUT_DIR)

def get_best_experiment(all_results: dict):
    best_name = None
    best_recall = -np.inf

    for name, result in all_results.items():
        summary_df = result["summary_df"].set_index("metric")
        mean_recall = summary_df.loc["recall", "mean"]

        if mean_recall > best_recall:
            best_recall = mean_recall
            best_name = name

    return best_name, all_results[best_name]


best_exp_name, best_exp_result = get_best_experiment(all_results)

print("\n" + "=" * 72)
print(f"BEST OVERALL EXPERIMENT: {best_exp_name.upper()}")
print("Best fold model info:")
print(best_exp_result["best_fold_model"])
print("=" * 72)

with open(Path(COMPARISON_OUT_DIR) / "best_experiment.json", "w") as f:
    json.dump(
        {
            "best_experiment": best_exp_name,
            "best_fold_model": best_exp_result["best_fold_model"],
        },
        f,
        indent=2
    )


  EXPERIMENT: BCE
  loss=bce  focal_alpha=0.25  focal_gamma=2.0  pos_weight=auto

========== Fold 1 ==========
Image path: /content/gdrive/MyDrive/CARDIUM/CARDIUM_image_embeddings/cardium_fetalclip_embeddings_fold_1.parquet
Exists? True
Tabular path: /content/gdrive/MyDrive/CARDIUM/CARDIUM_tabular_embeddings/cardium_tabular_embeddings_fold0.parquet
Exists? True
Loaded image embeddings: (6558, 777)
Image columns: ['image_path', 'fold', 'split', 'class_name', 'label', 'patient_id', 'image_id', 'filename', 'ok', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb

KeyError: 'best_fold_model'